In [1]:
import fine as fn
import pyomo.environ as pyomo


# Step 1: Define the Energy System Model
esM = fn.EnergySystemModel(
    locations={"A"},
    commodities={"electricity"},
    commodityUnitsDict={"electricity": "GW"},
    materials={"steel", "copper"},
    materialUnitsDict={"steel": "tons", "copper": "kg"}
)
esM.pyM = pyomo.ConcreteModel()


In [2]:
# Step 2: Add a Material Source (Raw Material Supplier)
esM.add(
    fn.Source(
        esM=esM, 
        name="Electricity",
        #materialConsumption="steel",
        commodity="electricity",
        hasCapacityVariable=True,
        materialConsumption={"steel": 2, "copper": 0.5},  # Materials required for commissioning
        materialRecovery={"steel": 0.8, "copper": 0.3}   # Recovery fractions at decommissioning
    )
)



In [3]:
esM.add(
    fn.Source(
        esM=esM, 
        name="Steel Supply",
        #materialConsumption="steel",
        materials="steel",
        hasCapacityVariable=True,
    )
)

In [4]:
# Step 2: Add a Material Source (Raw Material Supplier)
esM.add(
    fn.Source(
        esM=esM, 
        name="Steel Supply",
        #materialConsumption="steel",
        materials="steel",
        hasCapacityVariable=True,
    )
)


/fast/home/n-ludwig/model_Git/fine/fine/component.py:729: UserWarning: Component identifier Steel Supply already exists. Data will be overwritten.
  warnings.warn(


In [5]:
# Step 2: Add a Material Source (Raw Material Supplier)
esM.add(
    fn.Source(
        esM=esM, 
        name="Copper Supply",
        #materialConsumption="steel",
        materials="copper",
        hasCapacityVariable=True,
        #materialConsumption={"steel": 2, "copper": 0.5},  # Materials required for commissioning
        #materialRecovery={"steel": 0.8, "copper": 0.3}   # Recovery fractions at decommissioning
    )
)

In [6]:
# Step 4: Add a Storage Component that Requires Materials

esM.add(
    fn.Storage(
        esM=esM,
        name="Battery",
        commodity="electricity",
        chargeEfficiency=0.9,
        dischargeEfficiency=0.9,
        materialConsumption={"steel": 2, "copper": 0.5},  # Materials required for commissioning
        materialRecovery={"steel": 0.8, "copper": 0.3}   # Recovery fractions at decommissioning
    )
) 

In [7]:
esM.optimize()

Declaring sets, variables and constraints for SourceSinkModel
	declaring sets... 
	declaring variables... 
	declaring constraints... 
		(0.3661 sec)

Declaring sets, variables and constraints for StorageModel
	declaring sets... 
	declaring variables... 
	declaring constraints... 
		(1.9292 sec)

Declaring shared potential constraint...
		(0.0005 sec)

Declaring linked component quantity constraint...
		(0.0000 sec)

Declaring commodity balances...
		(0.1927 sec)

		(0.0000 sec)

Declaring material balance constraints...
ERROR: Rule failed when generating expression for Constraint
materialBalanceConstraint with index ('A', 'copper', 0, 0, 0): AttributeError:
'Storage' object has no attribute 'material'
ERROR: Constructing component 'materialBalanceConstraint' from data=None
failed:
        AttributeError: 'Storage' object has no attribute 'material'


AttributeError: 'Storage' object has no attribute 'material'

In [ ]:
def materialBalanceConstraint(pyM, loc, mat, ip):
    expr = (
        sum(
            mdl.getMaterialBalanceContribution(pyM, mat, loc, ip)
            for mdl in pyM.modelingClasses
        ) 
        <= self.processedMaterialBalanceLimit.get(ip, pd.DataFrame()).get(mat, {}).get(loc, 0)
    )
    print(f"Constraint[{loc},{mat},{ip}]: {expr}")  # Debug output
    return expr


In [ ]:

esM.declareCommodityBalanceConstraints(esM.pyM) 
